# Auto3D Quick Start

Get up and running with Auto3D in 5 minutes. This notebook demonstrates the essential features for generating low-energy 3D molecular conformers.

In [ ]:
import Auto3D
print(f"Auto3D version: {Auto3D.__version__}")

## 1. Generate Conformers from SMILES

The simplest way to generate conformers is using `smiles2mols` for small batches:

In [ ]:
from Auto3D import Auto3DOptions, smiles2mols

# Define some molecules
smiles_list = [
    "CCO",           # ethanol
    "c1ccccc1",      # benzene
    "CC(=O)O",       # acetic acid
]

# Configure Auto3D
config = Auto3DOptions(
    k=1,              # Get top-1 conformer per molecule
    use_gpu=False,    # Use CPU (set True for GPU)
)

# Generate conformers
mols = smiles2mols(smiles_list, config)
print(f"Generated {len(mols)} conformers")

## 2. Inspect the Results

Each molecule has optimized coordinates and energy information:

In [ ]:
for mol in mols:
    name = mol.GetProp("_Name")
    energy = float(mol.GetProp("E_tot"))  # Energy in Hartree
    energy_kcal = energy * 627.509        # Convert to kcal/mol
    
    print(f"{name}:")
    print(f"  Energy: {energy:.6f} Hartree ({energy_kcal:.2f} kcal/mol)")
    print(f"  Atoms: {mol.GetNumAtoms()}")

## 3. Visualize the Conformers

View the 3D structures:

In [ ]:
from rdkit.Chem import Draw

# 2D view
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(250, 200))

## 4. Processing Larger Datasets

For larger datasets (>150 molecules), use the `main` function with file input:

In [ ]:
import os
from Auto3D import Auto3DOptions, main

# Get path to example file
root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
input_path = os.path.join(root, "example/files/smiles.smi")

if __name__ == "__main__":
    config = Auto3DOptions(
        path=input_path,
        k=1,
        use_gpu=False,
    )
    output_path = main(config)
    print(f"Results saved to: {output_path}")

## 5. Multiple Conformers

Generate multiple conformers per molecule:

In [ ]:
# Get top-3 conformers per molecule
config = Auto3DOptions(k=3, use_gpu=False)
mols = smiles2mols(["CCCCC"], config)  # pentane

print(f"Generated {len(mols)} conformers for pentane")
for mol in mols:
    energy = float(mol.GetProp("E_tot")) * 627.509
    rel_e = float(mol.GetProp("E_rel(kcal/mol)"))
    print(f"  Relative energy: {rel_e:.2f} kcal/mol")

## 6. Using Energy Window

Keep all conformers within an energy window:

In [ ]:
# Keep conformers within 3 kcal/mol of minimum
config = Auto3DOptions(window=3.0, use_gpu=False)
mols = smiles2mols(["CCCCCC"], config)  # hexane

print(f"Found {len(mols)} conformers within 3 kcal/mol window")

## 7. Choosing a Model

Auto3D supports multiple neural network potentials:

In [ ]:
# AIMNET (default) - most versatile, supports charged molecules
config = Auto3DOptions(k=1, optimizing_engine="AIMNET", use_gpu=False)

# ANI2x - fast, well-validated
config = Auto3DOptions(k=1, optimizing_engine="ANI2x", use_gpu=False)

# ANI2xt - ultra-fast, good for tautomers
config = Auto3DOptions(k=1, optimizing_engine="ANI2xt", use_gpu=False)

print("Available engines: AIMNET, ANI2x, ANI2xt")

## Next Steps

- See `tutorial.ipynb` for more detailed examples
- See `single_point_energy.ipynb` for energy calculations
- See `tautomer.ipynb` for tautomer enumeration
- Read the [documentation](https://auto3d.readthedocs.io/) for full details